# Native AHR — live progress

Purpose: inspect the detached experiment without starting another training writer. Inputs: current immutable configuration, controller/training manifests and reports. Outputs: a compact current status table. Expected runtime: under five seconds. No checkpoint is written by this notebook.

Training runs in `tmux attach -t ahr-go-native-v4`; detach with **Ctrl-b, then d**. Checkpoints live in `checkpoints/ahr-native-v4-main/`. Exact resume uses notebook 04 with `AHR_NATIVE_V4_RUN=1` and unchanged configuration/source. A completed cycle may still fail the 50% target; read final outcomes and UNKNOWN counts separately.


In [ ]:
from pathlib import Path
import json,datetime as dt
from zoneinfo import ZoneInfo
from IPython.display import display,Markdown
project=Path.cwd().resolve()
while not (project/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if project.parent==project: raise RuntimeError('Run inside AHR Go workspace')
    project=project.parent
root=project/'reports/ahr_native_v4'
read=lambda name:json.loads((root/name).read_text()) if (root/name).is_file() else None
controller=read('pipeline_status.json') or {'status':'NOT_STARTED'}
main_training=read('ahr-native-v4-main_training.json')
training=main_training or read('ahr-native-v4-launch-pilot-v3_training.json') or {}
if not main_training and training:
    controller={'status':'VALIDATION_PILOT_RUNNING','phase':training.get('phase'),'development':[]}
now=dt.datetime.now(ZoneInfo('Europe/Dublin')).isoformat(timespec='seconds')
rows=[('Checked (Dublin)',now),('Controller',controller.get('status')),('Phase',controller.get('phase','pending')),
      ('Training games',str(training.get('games_finished',0))+' / '+str(training.get('scheduled_games',2048))),
      ('Training run',training.get('run_id','pending')),('Optimizer steps',training.get('optimizer_steps',0)),('Active game plies',training.get('current_game_plies')),
      ('Training UNKNOWN',training.get('unknown_games',0)),('Session','ahr-go-native-v4')]
display(Markdown('| Item | Current value |\n|---|---|\n'+'\n'.join('| '+str(a)+' | '+str(b)+' |' for a,b in rows)))


In [ ]:
development=controller.get('development',[])
if development:
    lines=['| Evaluation | Wins | Losses | UNKNOWN | Wins / attempts |','|---|---:|---:|---:|---:|']
    for receipt in development:
        lines.append('| '+str(receipt.get('eval_run'))+' | '+str(receipt.get('wins'))+' | '+str(receipt.get('losses'))+' | '+str(receipt.get('unknown'))+' | '+format(receipt.get('wins_over_planned',0),'.1%')+' |')
    display(Markdown('\n'.join(lines)))
final=controller.get('final')
if final:
    display(Markdown('Final: **'+str(final['wins'])+' wins / '+str(final['games_planned'])+' attempts**; UNKNOWN: '+str(final['unknown'])+'. Observed half wins: '+str(final['observed_half_wins'])+'.'))
if controller.get('last_error'): display(Markdown('Last error: `'+controller['last_error'].replace('`','')+'`'))
